In [1]:
import requests
from bs4 import BeautifulSoup
import re
from datetime import datetime

In [2]:
url='https://www.cpc.gov.mo/api/news/list'
headers={'USER-AGENT': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/128.0.0.0 Safari/537.36 Edg/128.0.0.0'}
r=requests.get(url, headers=headers)
r

<Response [200]>

In [3]:
data=r.json()['data']
restructured_posts=[]
domain='https://www.cpc.gov.mo/'

for post in data['list']:
    title=post['title']
    if post.get('url'):
        link=domain+post['url']
    else:
        link=''
    date=post['dateAt']
    # date=datetime.strptime(date, '%Y-%m-%dT%H:%M:%S%z').strftime('%Y-%m-%d %H:%M:%S')
    if post.get('content'):
        content=BeautifulSoup(BeautifulSoup(post['content'], 'html.parser').decode('utf-8')).text
    else:
        content=''
    dict_post={
            'title': title,
            'link': link,
            'date': date,
            'content': content
        }
    restructured_posts.append(dict_post)

In [4]:

import xml.etree.ElementTree as ET

# Create the root of the RSS feed
rss = ET.Element('rss', version='2.0')
channel = ET.SubElement(rss, 'channel')

# Add channel elements
ET.SubElement(channel, 'title').text = 'Comissão Profissional dos Contabilistas Notícias'
ET.SubElement(channel, 'link').text = url
ET.SubElement(channel, 'description').text = 'CPC RSS'

# Add items
for item in restructured_posts:
    item_elem = ET.SubElement(channel, 'item')
    ET.SubElement(item_elem, 'title').text = item['title']
    ET.SubElement(item_elem, 'link').text = item['link']
    ET.SubElement(item_elem, 'description').text = item['content']
    ET.SubElement(item_elem, 'pubDate').text = item['date']

# Convert to string and save to an XML file
rss_feed = ET.tostring(rss, encoding='utf-8', method='xml').decode()
with open('CPC.xml', 'w', encoding='utf-8') as xml_file:
    xml_file.write(rss_feed)

In [5]:
import base64

# Variables
token = 'Replaced by real toekn'
repo = 'mogcai/CPC_RSS'  # Replace with your GitHub username/repo
commit_message = 'Update RSS feed'
file_path='CPC.xml'

# Read the file content
with open(file_path, 'rb') as f:
    content = f.read()


In [6]:

# Encode the content
encoded_content = base64.b64encode(content).decode()
github_path='CPC.xml'
# Prepare the API endpoint URL
url = f'https://api.github.com/repos/{repo}/contents/{github_path}'



In [7]:

# Fetch the current SHA of the file (if it exists)
response = requests.get(url, headers=headers)
if response.status_code == 200:
    file_info = response.json()
    sha = file_info['sha']  # Get the SHA of the file
else:
    sha = None  # File does not exist
    print(f"Error fetching file info: {response.content.decode()}")


In [8]:

# Prepare the payload
payload = {
    'message': commit_message,
    'content': encoded_content,
    'branch': 'main',  # Change to 'master' if that is your default branch
}

if sha:
    payload['sha'] = sha  # Include the SHA if the file exists



# Make the API request to create or update the file
headers = {
    'Authorization': f'token {token}',
    'Accept': 'application/vnd.github.v3+json'
}


In [9]:
import json
response = requests.put(url, headers=headers, data=json.dumps(payload))

# Check the response
if response.status_code == 201:
    print('File uploaded successfully.')
elif response.status_code == 200:
    print('File updated successfully.')
else:
    print(f'Error: {response.content}')


File updated successfully.
